In [0]:
import importlib
import config as C
import citibike_lib as L
from pyspark.sql import functions as F
from pyspark.sql.window import Window

importlib.reload(C)
importlib.reload(L)
L.init_spark(spark)

OUTPUT_TABLE = C.fq("weather_model_examples")

forecast_history = spark.table(C.WEATHER_FORECAST_TABLE)

weather_start = forecast_history.agg(
    F.min("retrieved_at_utc")
).first()[0]

# Load the station IDs selected for the current 100-station MVP.
mvp_ids = spark.table(C.T["mvp_stations"]).select("station_id")

print("Output table:", OUTPUT_TABLE)
print("Forecast collection began:", weather_start)
print("Selected stations:", mvp_ids.count())

In [0]:
# Load recent bike observations for MVP stations, including history needed for lag features.
import datetime as dt

history_minutes = (
    max(C.LAG_STEPS) * C.GRID_MIN
    + C.ASOF_TOLERANCE_MIN
)

if weather_start is None:
    raise ValueError("No saved forecasts were found.")

read_start = weather_start - dt.timedelta(minutes=history_minutes)

bike_observations = (
    spark.table("workspace.default.live_station_status")
    .where(F.col("fetched_at") >= F.lit(read_start))
    .join(mvp_ids, on="station_id", how="inner")
    .withColumnRenamed("fetched_at", "snapshot_ts")
    .withColumnRenamed("station_reported_at", "last_reported")
)

for column in ["is_installed", "is_renting", "is_returning"]:
    bike_observations = bike_observations.withColumn(
        column, F.col(column).cast("int")
    )

display(
    bike_observations.agg(
        F.count("*").alias("rows"),
        F.countDistinct("station_id").alias("stations"),
        F.min("snapshot_ts").alias("first_observation"),
        F.max("snapshot_ts").alias("latest_observation")
    )
)

In [0]:
# Select one latest station-information record per MVP station.
latest_info_window = Window.partitionBy("station_id").orderBy(
    F.col("fetched_at").desc()
)

station_info = (
    spark.table("workspace.default.live_station_info")
    .join(mvp_ids, on="station_id", how="inner")
    .withColumn(
        "row_number",
        F.row_number().over(latest_info_window)
    )
    .where(F.col("row_number") == 1)
    .select("station_id", "name", "capacity", "lat", "lon")
)

print("Stations with metadata:", station_info.count())

In [0]:
# Build bike features and 15-minute outcome labels for the forecast collection period.
bike_examples = L.build_features_labels(
    bike_observations,
    station_info,
    C,
    with_labels=True
)

bike_examples = bike_examples.where(
    F.col("prediction_time_utc") >= F.lit(weather_start)
)

display(
    bike_examples.groupBy("eligible", "exclusion_reason")
    .count()
    .orderBy(F.desc("count"))
)

In [0]:
# Attach previously collected forecasts to bike examples and check timing and coverage.
weather_examples = L.add_weather(bike_examples)

print("Bike example rows:", bike_examples.count())

display(
    weather_examples.agg(
        F.count("*").alias("output_rows"),
        F.count("weather_retrieved_at_utc").alias("rows_with_forecast"),
        F.sum(
            F.when(
                (F.col("eligible") == 1)
                & F.col("weather_retrieved_at_utc").isNotNull(),
                1
            ).otherwise(0)
        ).alias("eligible_rows_with_forecast"),
        F.sum(
            F.when(
                F.col("weather_retrieved_at_utc")
                > F.col("prediction_time_utc"),
                1
            ).otherwise(0)
        ).alias("forecasts_retrieved_too_late")
    )
)

In [0]:
# Save the weather-enriched examples to a separate Delta table, replacing its previous contents.
(
    weather_examples.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(OUTPUT_TABLE)
)

print("Saved table:", OUTPUT_TABLE)
print("Saved rows:", spark.table(OUTPUT_TABLE).count())